# A first look at SpatialLeiden on human myocardial infarction (Kuppe et al. 2022)

This notebook is a gentle introduction to **SpatialLeiden** for people who already know scanpy. We use one Visium section from Kuppe et al. 2022 (*Nature* 608, 766–777) and compare ordinary Leiden clustering with SpatialLeiden.

**What SpatialLeiden does.** Ordinary Leiden clusters spots using one graph: the k-nearest-neighbour (kNN) graph of gene expression. SpatialLeiden adds a second graph that links spots that are physical neighbours on the slide, and runs *multiplex* Leiden on both graphs at once. A single partition is optimised to be good on both layers. The parameter `layer_ratio` sets how much the spatial layer counts relative to the expression layer: 0 means plain Leiden, larger values give more spatially continuous domains.

**Plan.**

1. Load one section and the authors' annotations.
2. Look at the tissue and at the authors' niches.
3. Preprocess as in the SpatialLeiden preprint.
4. Build the expression graph and the spatial graph.
5. Run Leiden and SpatialLeiden with the same number of clusters.
6. Compare the results with the authors' niches and measure spatial coherence.

## About this notebook

| | |
|---|---|
| **Purpose** | A first, hands-on look at SpatialLeiden for hackathon participants who know scanpy. One slide, four clusterings, a small comparison table. |
| **Data** | Kuppe et al. 2022, Visium section `GT/IZ_P9_rep2` (run `ACH0013`, patient P9, ischaemic zone; the P9 slide in the paper's Fig. 2b). Raw counts from the Zenodo h5ad (`data/kuppe_2022/processed1/`), niches and coordinates from the per-spot table extracted from the authors' Seurat object (`data/kuppe_2022/visium_tables/`). Both are git-ignored; see `downloads/kuppe_2022.py` and `01_data_preprocess/kuppe_visium_exploration/`. |
| **Environment** | Conda env `spatialleiden` (kernel "Python (spatialleiden)"): scanpy 1.10.4, squidpy 1.5.0, anndata 0.10.9, spatialleiden 0.4.x, scikit-learn. |
| **How to run** | Open it from `Derived_Modalities/03_single_modality_clustering/` with the `spatialleiden` kernel and run all cells. Figures and the results table are written to `Derived_Modalities/results/kuppe_2022/intro_notebook/`. |
| **Runtime** | About one minute on our VM (32 CPUs); most of it is the resolution search. |

### Data traps (from `01_data_preprocess/kuppe_visium_exploration/NOTES.md`)

- **Rescaled coordinates.** The h5ad coordinates (`obsm["X_spatial"]`) are rescaled to 0–1 separately along x and y, which distorts the hexagonal grid. We take the true pixel coordinates from the authors' Seurat object instead.
- **No niches in the h5ad.** The published cell-type and molecular niches exist only in the authors' Seurat objects, so we join the two sources **on the spot barcode** and check that every spot matches.
- **High mitochondrial fraction.** Healthy myocardium has 31–40 % mitochondrial reads per spot; a standard 20 % filter would remove most of it. We do not filter on it here.
- **Different sample names.** The region name (`GT/IZ_P9_rep2`), the h5ad file name (`Visium_GT_IZ_P9_rep2.h5ad`) and the run ID (`ACH0013`) are linked through `visium_raw/metadata-Visium.csv`.

**Extended on 23 November 2026** with sections 7–14 (experiment 0): the `layer_ratio` sweep, the choice of the number of domains K, and the sanity check of the no-space baseline. These sections add about 2 minutes. The full sweep (3 slides × 8 ratios × 5 seeds × 4 K settings) is loaded from `results/kuppe_2022/` if present, or recomputed in parallel through `exp0_layer_ratio_sweep/exp0_sweep.py` (about 1–2 minutes per K setting).

## 0. Setup

We import the usual single-cell packages plus `spatialleiden` (imported as `sl`). One random seed is used everywhere so the results are reproducible.

In [ ]:
import json
import time
import warnings

# silence two import-time warnings from dask and pkg_resources (harmless, they only clutter the output)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*pkg_resources.*")
from pathlib import Path  # noqa: E402

import anndata as ad  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import scanpy as sc  # noqa: E402
import spatialleiden as sl  # noqa: E402
import squidpy as sq  # noqa: E402
from scipy.optimize import linear_sum_assignment  # noqa: E402
from sklearn.metrics import (  # noqa: E402
    adjusted_rand_score,
    normalized_mutual_info_score,
)

random_state = 42
t_start = time.time()
print(
    "spatialleiden",
    sl.__version__,
    "| scanpy",
    sc.__version__,
    "| squidpy",
    sq.__version__,
)

Next we locate the data. The region name is translated to the run ID through `metadata-Visium.csv` (see the data traps above). The paths assume the notebook sits in `Derived_Modalities/03_single_modality_clustering/`. The data are git-ignored and live under `Derived_Modalities/data/kuppe_2022/`; figures are written to `Derived_Modalities/results/kuppe_2022/intro_notebook/`.

In [ ]:
# find the Derived_Modalities folder, whether we run from 03_.../ or from Derived_Modalities/
here = Path.cwd().resolve()
DM = next(p for p in [here, *here.parents] if p.name == "Derived_Modalities")
DATA = DM / "data" / "kuppe_2022"
OUT = DM / "results" / "kuppe_2022" / "intro_notebook"
OUT.mkdir(parents=True, exist_ok=True)

REGION = "GT/IZ_P9_rep2"  # the ischaemic P9 slide shown in Kuppe Fig. 2b
meta = pd.read_csv(DATA / "visium_raw" / "metadata-Visium.csv")
run_id = meta.loc[meta["patient_region_id"] == REGION, "hca_sample_id"].item()
h5ad_path = DATA / "processed1" / f"Visium_{REGION.replace('/', '_')}.h5ad"
print(REGION, "->", run_id, "|", h5ad_path.name)

## 1. Load one section

Two sources are combined:

- the Zenodo h5ad (`processed1/`), which holds the **raw counts** in `.raw` (`X` there is already log-normalised);
- a per-spot table extracted from the authors' Seurat object, which holds the published **cell-type niches**, **molecular niches** and the **true spot coordinates**.

The h5ad coordinates (`obsm["X_spatial"]`) are rescaled to 0–1 separately along x and y, which distorts the hexagonal grid, so we do not use them.

In [ ]:
h5 = ad.read_h5ad(h5ad_path)
spots = pd.read_csv(DATA / "visium_tables" / f"{run_id}_spots.csv")
print(h5)
print("spot table:", spots.shape)
print("h5ad obs_names:", list(h5.obs_names[:3]))
print("table barcodes:", list(spots["barcode"][:3]))

**How to read this:** the h5ad has 4,113 spots and 17,708 genes; the spot table has one row per spot. Look at the first barcodes printed: both use the same format (`AAAC…-1`), with no sample prefix or suffix.

So we join on the barcode and check that **every** spot matches in both directions. We also check that the raw counts add up to the authors' per-spot UMI totals, which confirms `.raw` really holds raw counts.

In [ ]:
spots = spots.set_index("barcode")
n_h5, n_tab = h5.n_obs, len(spots)
n_match = len(h5.obs_names.intersection(spots.index))
print(f"h5ad spots: {n_h5}, table spots: {n_tab}, matched: {n_match}")
assert n_match == n_h5 == n_tab, "barcodes do not match one-to-one"
spots = spots.loc[h5.obs_names]

# sanity check: the raw counts in the h5ad add up to the authors' nCount_Spatial
raw_total = np.asarray(h5.raw.X.sum(axis=1)).ravel()
assert np.allclose(raw_total, spots["nCount_Spatial"].to_numpy())

Now we build a clean AnnData: raw counts in `X` (and a copy in `layers["counts"]`), the authors' log-normalised values in a layer for reference, the true pixel coordinates in `obsm["spatial"]` (x = column, y = row), the array grid position, and the authors' labels. Zones are given per section, not per spot, so `zone` is the same for every spot here.

In [ ]:
adata = ad.AnnData(
    X=h5.raw.X.copy(),
    obs=pd.DataFrame(index=h5.obs_names),
    var=pd.DataFrame(index=h5.raw.var_names),
)
adata.layers["counts"] = adata.X.copy()
assert list(h5.var_names) == list(h5.raw.var_names), "X and .raw differ in gene order"
adata.layers["authors_lognorm"] = h5.X.copy()  # same genes, same order as .raw (asserted above)
adata.obsm["spatial"] = spots[["pxl_col_fullres", "pxl_row_fullres"]].to_numpy(float)

obs_cols = [
    "array_row",
    "array_col",
    "nCount_Spatial",
    "nFeature_Spatial",
    "percent.mt",
    "celltype_niche",
    "molecular_niche",
    "patient_region_id",
    "major_labl",
]
for c in obs_cols:
    adata.obs[c] = spots[c].to_numpy()
adata.obs = adata.obs.rename(columns={"major_labl": "zone"})

# order niche categories numerically (ctniche_1 ... ctniche_9, molniche_1 ... molniche_12)
for c in ["celltype_niche", "molecular_niche"]:
    cats = sorted(adata.obs[c].unique(), key=lambda s: int(s.split("_")[1]))
    adata.obs[c] = pd.Categorical(adata.obs[c], categories=cats)

adata

## 2. The tissue and the authors' niches

First the low-resolution H&E image with the spots on top. Spot pixel coordinates refer to the full-resolution image, so we multiply them by `tissue_lowres_scalef` from the Space Ranger scale factors.

In [ ]:
img = plt.imread(DATA / "visium_tables" / f"{run_id}_lowres.png")
scalef = json.loads(
    (DATA / "visium_tables" / f"{run_id}_scalefactors.json").read_text()
)
xy_low = adata.obsm["spatial"] * scalef["tissue_lowres_scalef"]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, show_spots in zip(axes, [False, True]):
    ax.imshow(img)
    if show_spots:
        ax.scatter(*xy_low.T, s=3, c="#0072B2", alpha=0.8, linewidths=0)
    ax.set_xlabel("x (low-res pixels)")
    ax.set_ylabel("y (low-res pixels)")
axes[0].set_title("H&E (low resolution)")
axes[1].set_title(f"{adata.n_obs:,} spots on tissue")
fig.tight_layout()
fig.savefig(OUT / "fig1_he_spots.png", dpi=150)

Next, the authors' labels. **Cell-type niches** group spots by their estimated cell-type composition (cell2location); **molecular niches** group spots by gene expression. Both were made **without** spot coordinates, so any spatial structure in them comes from the tissue itself. We use the authors' niche colours and a small plotting helper for all maps in this notebook.

In [ ]:
NICHE_COLORS = {
    f"ctniche_{i}": c
    for i, c in enumerate(
        [
            "#D51F26",
            "#272E6A",
            "#208A42",
            "#89288F",
            "#F47D2B",
            "#FEE500",
            "#8A9FD1",
            "#C06CAB",
            "#D8A767",
        ],
        start=1,
    )
}
# 12 distinguishable colours for the molecular niches (Paul Tol "muted" + 3 Okabe-Ito)
MOL_PALETTE = [
    "#CC6677",
    "#332288",
    "#DDCC77",
    "#117733",
    "#88CCEE",
    "#882255",
    "#44AA99",
    "#999933",
    "#AA4499",
    "#000000",
    "#E69F00",
    "#56B4E9",
]
MOL_COLORS = dict(zip(adata.obs["molecular_niche"].cat.categories, MOL_PALETTE))


def spatial_map(ax, labels, colors, title, s=4, legend=True):
    # plot spots at their full-resolution pixel position (y axis points down, as in the image)
    xy = adata.obsm["spatial"]
    labels = pd.Series(labels, index=adata.obs_names).astype(str)
    for lab in [k for k in colors if k in set(labels)]:
        m = (labels == lab).to_numpy()
        ax.scatter(xy[m, 0], xy[m, 1], s=s, c=colors[lab], label=lab, linewidths=0)
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(title, fontsize=10)
    if legend:
        ax.legend(
            loc="upper left",
            bbox_to_anchor=(1.0, 1.0),
            frameon=False,
            markerscale=2.5,
            fontsize=7,
            handletextpad=0.2,
        )


fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
spatial_map(
    axes[0], adata.obs["celltype_niche"], NICHE_COLORS, "Authors: cell-type niches"
)
spatial_map(
    axes[1], adata.obs["molecular_niche"], MOL_COLORS, "Authors: molecular niches"
)
fig.tight_layout()
fig.savefig(OUT / "fig2_author_niches.png", dpi=150)

print(adata.obs["celltype_niche"].value_counts().sort_index().to_string())

**How to read this:** each dot is one Visium spot, coloured by the authors' label; the list below the maps gives the number of spots per cell-type niche. Note that some niches form large patches while others are scattered. Niche numbering follows the authors' `ctniche_1…9`; its match to the paper's "niche 1–9" in Fig. 2c is still to be checked (handoff, step 7.2).

We will ask every clustering for as many clusters as there are cell-type niches on this slide. Comparing runs with the same number of clusters keeps the comparison fair.

In [ ]:
n_clusters = adata.obs["celltype_niche"].nunique()
print("cell-type niches on this slide:", n_clusters)
print("molecular niches on this slide:", adata.obs["molecular_niche"].nunique())

## 3. Preprocessing (as in the SpatialLeiden preprint)

The preprint's recipe: keep genes detected in at least 10 spots, select 2,000 highly variable genes with `seurat_v3` (which works on raw counts), normalise, log-transform, scale **without** zero-centring, and compute 30 principal components.

A note on quality control. Myocardium has a very high mitochondrial fraction (median 31–40 % per spot in control, remote and border-zone slides; see `01_data_preprocess/kuppe_visium_exploration/NOTES.md`, item 12), so the usual 20 % filter would remove most heart muscle. We do **not** filter spots here. (The h5ad contains no `MT-` genes, so the fraction shown below is the authors' `percent.mt`.)

The next cell just prints the distribution of the mitochondrial fraction on this slide.

In [ ]:
print(adata.obs["percent.mt"].describe().round(1).to_string())

**How to read this:** this ischaemic slide has a *low* mitochondrial fraction (median 6 %), while the control slide P1 has 32 %. That is the opposite of what one might expect from damaged tissue. The likely reason is composition: cardiomyocytes are packed with mitochondria, so cardiomyocyte-rich healthy myocardium has a high mitochondrial share, and an infarct that has lost cardiomyocytes has a low one. Handling of the donor hearts may add to it, but the data cannot separate the two. Either way, a fixed cut-off would remove spots for biological reasons, so we keep all of them. It also means that on P1, the negative control used later, data quality and tissue type are confounded.

Now the preprocessing itself. All parameters follow the SpatialLeiden 2 preprint (Methods, "Preprocessing"): `filter_genes(min_cells=10)`, 2,000 HVGs with `flavor="seurat_v3"` on the raw counts (layer `counts`), `normalize_total`, `log1p`, `scale(zero_center=False)` and `pca(n_comps=30)` on the HVGs. Scaling without centring keeps the matrix sparse and is what the preprint uses for single slides.

Two choices to revisit at the multi-sample step:

- **`normalize_total` without `target_sum`** scales each spot to the slide's median count, as in the preprint. Each slide is clustered on its own here, so this does not affect the comparisons. For pooled slides we will fix `target_sum=1e4`.
- **`filter_genes`** runs per slide, so slides keep slightly different gene sets. Before pooling slides, the gene set must be harmonised.

In [ ]:
sc.pp.filter_genes(adata, min_cells=10)
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat_v3", layer="counts")
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
sc.pp.scale(adata, zero_center=False)
sc.pp.pca(adata, n_comps=30, mask_var="highly_variable", random_state=random_state)
print(
    adata.n_vars,
    "genes kept,",
    int(adata.var["highly_variable"].sum()),
    "HVGs; PCA:",
    adata.obsm["X_pca"].shape,
)

**How to read this:** the gene filter removes nothing on this slide (all 17,708 genes in the deposited h5ad are detected in at least 10 spots), and PCA runs on the 2,000 HVGs.

## 4. Two graphs: expression and space

**Expression graph.** The standard scanpy kNN graph on the 30 PCs with 15 neighbours (preprint Methods, "Neighbourhood graph generation"). It is stored in `obsp["connectivities"]`, which is where SpatialLeiden looks for it by default.

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30, random_state=random_state)

**Spatial graph.** Visium spots sit on a hexagonal grid, so each spot has up to six direct neighbours. `squidpy.gr.spatial_neighbors` with `coord_type="grid"` and `n_neighs=6` builds exactly this graph and stores it in `obsp["spatial_connectivities"]`. This is the setting the preprint uses for Visium (Methods, "Neighbourhood graph generation") and the package's basics guide recommends for Visium. All edges get the same weight because grid neighbours are equidistant.

In [ ]:
sq.gr.spatial_neighbors(adata, coord_type="grid", n_neighs=6)
A = adata.obsp["spatial_connectivities"]
degree = np.asarray((A > 0).sum(axis=1)).ravel()
print("spatial neighbours per spot:")
print(pd.Series(degree).value_counts().sort_index().to_string())

**How to read this:** the table counts spots by their number of spatial neighbours. Most spots have six; spots at the tissue edge or next to holes have fewer.

As a check, we also list each spot's hexagonal neighbours directly from the array grid (offsets (0, ±2) and (±1, ±1) in `array_row`/`array_col`) and confirm that squidpy found the same pairs. We reuse this neighbour list later to measure spatial coherence.

In [ ]:
HEX = [(0, 2), (0, -2), (1, 1), (1, -1), (-1, 1), (-1, -1)]
pos = {
    (r, c): i
    for i, (r, c) in enumerate(zip(adata.obs["array_row"], adata.obs["array_col"]))
}
nbr_i, nbr_j = [], []
for (r, c), i in pos.items():
    for dr, dc in HEX:
        j = pos.get((r + dr, c + dc))
        if j is not None:
            nbr_i.append(i)
            nbr_j.append(j)
nbr_i, nbr_j = np.array(nbr_i), np.array(nbr_j)

grid_pairs = set(zip(nbr_i, nbr_j))
sq_pairs = set(zip(*A.nonzero()))
print(
    f"array-grid pairs: {len(grid_pairs)}, squidpy pairs: {len(sq_pairs)}, identical: {grid_pairs == sq_pairs}"
)

**How to read this:** `identical: True` means squidpy's graph and the array grid agree exactly, so the pixel coordinates we took from the Seurat object are correct.

Here is what one spatial neighbourhood looks like: a spot in the middle of the tissue (black) and its six neighbours (orange).

In [ ]:
xy = adata.obsm["spatial"]
centre = int(
    np.argmin(((xy - np.median(xy, axis=0)) ** 2).sum(1) + 1e12 * (degree != 6))
)
nbrs = A[centre].nonzero()[1]
d = np.abs(xy - xy[centre]).max(1) < 600

fig, ax = plt.subplots(figsize=(3.6, 3.6))
ax.scatter(*xy[d].T, s=60, c="#BBBBBB", linewidths=0)
ax.scatter(*xy[nbrs].T, s=60, c="#E69F00", linewidths=0, label="neighbours")
ax.scatter(*xy[centre], s=60, c="black", linewidths=0, label="spot")
ax.set_aspect("equal")
ax.invert_yaxis()
ax.set_xlabel("x (full-res pixels)")
ax.set_ylabel("y (full-res pixels)")
ax.legend(frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(1, 1))
ax.set_title("One spatial neighbourhood", fontsize=10)
fig.tight_layout()

## 5. Leiden versus SpatialLeiden

We run four clusterings, all with **9 clusters** (the number of cell-type niches on this slide):

- **Leiden:** expression graph only (`spatialleiden.leiden`, with the same settings as SpatialLeiden).
- **SpatialLeiden** with `layer_ratio` = 0.5, 1 and 2.

**How we get exactly 9 clusters**, in two explicit steps (the same steps `spatialleiden.search_resolution` takes internally):

1. `sl.search_resolution_latent` finds, **once**, the expression-layer resolution that gives 9 clusters with plain Leiden.
2. For each `layer_ratio`, `sl.search_resolution_spatial` keeps that expression resolution fixed and adjusts the spatial-layer resolution until SpatialLeiden also gives 9.

**Other settings:**

- Following the preprint code (`clustering/spatialleiden_multisample.py`), both graphs are treated as undirected (`directed=False`); both are symmetric here.
- The `layer_ratio` values bracket the preprint's choice for single Visium slides (0.8; Methods, "Visium dorsolateral prefrontal cortex").
- We raise the search's `n_iter` from 15 to 30, so it does not stop early, and fix `random_state=42`.

In [ ]:
LAYER_RATIOS = [0.5, 1.0, 2.0]
latent_kwargs = {"directed": False, "random_state": random_state}

# step 1: the expression resolution giving n_clusters with plain Leiden, searched once
t0 = time.time()
res_latent = sl.search_resolution_latent(
    adata, n_clusters, n_iter=30, key_added="leiden", **latent_kwargs
)
sl.leiden(adata, resolution=res_latent, key_added="leiden", **latent_kwargs)
print(
    f"leiden: resolution={res_latent:.3f}, clusters={adata.obs['leiden'].nunique()}  "
    f"({time.time() - t0:.1f}s)"
)
runs = {"leiden": (res_latent, None, 0.0)}  # name -> (latent res, spatial res, layer_ratio)

# step 2: for each layer_ratio, keep res_latent and search the spatial resolution
for ratio in LAYER_RATIOS:
    key = f"sl_ratio_{ratio:g}"
    t0 = time.time()
    res_spatial = sl.search_resolution_spatial(
        adata,
        n_clusters,
        n_iter=30,
        resolution=(res_latent, 1.0),
        directed=False,
        layer_ratio=ratio,
        random_state=random_state,
        key_added=key,
    )
    # run once more at the chosen resolutions so the stored labels match them exactly
    sl.spatialleiden(
        adata,
        resolution=(res_latent, res_spatial),
        layer_ratio=ratio,
        directed=False,
        random_state=random_state,
        key_added=key,
    )
    runs[key] = (res_latent, res_spatial, ratio)
    print(
        f"{key}: resolution latent={res_latent:.3f}, spatial={res_spatial:.3f}, "
        f"clusters={adata.obs[key].nunique()}  ({time.time() - t0:.1f}s)"
    )

for key in runs:
    assert adata.obs[key].nunique() == n_clusters, (
        f"{key} did not reach {n_clusters} clusters"
    )

**How to read this:**

- **Every line confirms 9 clusters.**
- **The expression resolution is the same in every run.** It was searched once, with plain Leiden, and passed explicitly to every SpatialLeiden run.
- **Only the spatial resolution changes** with `layer_ratio`.

Section 10 shows a subtlety: SpatialLeiden at `layer_ratio = 0` is not exactly plain Leiden. That is why the systematic sweep in sections 11–12 uses a ratio-0 baseline made with SpatialLeiden itself.

Cluster numbers are arbitrary, so to make the maps easy to compare we match each cluster to the cell-type niche it overlaps most (one-to-one, Hungarian matching on the overlap table) and give it that niche's colour. This only changes names and colours, not the clustering.

In [ ]:
def match_to_niches(labels, ref):
    tab = pd.crosstab(pd.Series(labels, name="cl"), pd.Series(ref, name="ref"))
    rows, cols = linear_sum_assignment(-tab.to_numpy())
    mapping = {tab.index[r]: tab.columns[c] for r, c in zip(rows, cols)}
    return pd.Series(labels).map(mapping).to_numpy()


ref = adata.obs["celltype_niche"].astype(str).to_numpy()
for key in runs:
    adata.obs[f"{key}_matched"] = match_to_niches(
        adata.obs[key].astype(str).to_numpy(), ref
    )

titles = {
    "leiden": "Leiden (expression only)",
    **{f"sl_ratio_{r:g}": f"SpatialLeiden, layer_ratio = {r:g}" for r in LAYER_RATIOS},
}

fig, axes = plt.subplots(1, 5, figsize=(19, 4.2))
spatial_map(
    axes[0],
    adata.obs["celltype_niche"],
    NICHE_COLORS,
    "Authors: cell-type niches",
    s=2.5,
    legend=False,
)
for ax, key in zip(axes[1:], runs):
    spatial_map(
        ax, adata.obs[f"{key}_matched"], NICHE_COLORS, titles[key], s=2.5, legend=False
    )
handles = [
    plt.Line2D(
        [], [], ls="", marker="o", color=c, label=k.replace("ctniche_", "niche ")
    )
    for k, c in NICHE_COLORS.items()
]
fig.legend(handles=handles, loc="lower center", ncol=9, frameon=False, fontsize=9)
fig.tight_layout(rect=(0, 0.07, 1, 1))
fig.savefig(OUT / "fig3_leiden_vs_spatialleiden.png", dpi=150)

**How to read this:** the left map is the authors' reference; the other four are our clusterings, coloured by the niche they overlap most. Watch how the maps get smoother, with fewer isolated spots, from left to right.

## 6. How do the runs compare?

We report three numbers per clustering:

- **ARI and NMI** (adjusted Rand index, normalised mutual information) against the authors' cell-type niches and against their molecular niches. Both are 1 for identical partitions and about 0 for unrelated ones. The authors' niches are a reference, not ground truth: they were made from composition or expression alone, without space.
- **Spatial coherence:** the share of **neighbouring spot pairs** (edges of the spatial grid) whose two spots carry the same label. This is the join-count form, and the same definition is used throughout the notebook. We compare it with the same number after shuffling the labels across spots (mean of 20 shuffles, fixed seed), which keeps the cluster sizes but removes any spatial structure.

In [ ]:
def coherence(labels):
    """Edge-weighted (join-count form): share of neighbouring spot pairs with the same label."""
    labels = np.asarray(labels)
    return float((labels[nbr_i] == labels[nbr_j]).mean())


def shuffled_coherence(labels, n=20):
    """Mean coherence after shuffling labels. A fresh generator with a fixed seed per call,
    so the result does not depend on how many runs were scored before."""
    rng = np.random.default_rng(0)
    labels = np.asarray(labels)
    return float(np.mean([coherence(rng.permutation(labels)) for _ in range(n)]))


ct = adata.obs["celltype_niche"].astype(str).to_numpy()
mol = adata.obs["molecular_niche"].astype(str).to_numpy()

rows = []
for name, labels in [
    ("authors: cell-type niches", ct),
    ("authors: molecular niches", mol),
]:
    rows.append(
        {
            "run": name,
            "layer_ratio": np.nan,
            "n_clusters": len(set(labels)),
            "coherence": coherence(labels),
            "coherence_shuffled": shuffled_coherence(labels),
        }
    )
for key, (r_lat, r_sp, ratio) in runs.items():
    lab = adata.obs[key].astype(str).to_numpy()
    rows.append(
        {
            "run": titles[key],
            "layer_ratio": ratio,
            "n_clusters": len(set(lab)),
            "res_latent": r_lat,
            "res_spatial": r_sp,
            "ARI_ct": adjusted_rand_score(ct, lab),
            "NMI_ct": normalized_mutual_info_score(ct, lab),
            "ARI_mol": adjusted_rand_score(mol, lab),
            "NMI_mol": normalized_mutual_info_score(mol, lab),
            "coherence": coherence(lab),
            "coherence_shuffled": shuffled_coherence(lab),
        }
    )

results = pd.DataFrame(rows).set_index("run")
results = results[
    [
        "layer_ratio",
        "n_clusters",
        "res_latent",
        "res_spatial",
        "ARI_ct",
        "NMI_ct",
        "ARI_mol",
        "NMI_mol",
        "coherence",
        "coherence_shuffled",
    ]
]
results.to_csv(OUT / "results_table.csv")
results.round(3)

**How to read this:** `res_latent` / `res_spatial` are the resolutions used; `_ct` columns compare with the cell-type niches and `_mol` columns with the molecular niches. For coherence, the gap between `coherence` and `coherence_shuffled` is what matters: the shuffled value is what you get by chance with the same cluster sizes. The two "authors" rows give the coherence of the reference labels themselves. The table is also saved as `results_table.csv`.

The same numbers as a picture: agreement with the authors' niches (left) and spatial coherence (right), as `layer_ratio` grows from 0 (plain Leiden) to 2.

In [ ]:
res = results.dropna(subset=["ARI_ct"])
x = res["layer_ratio"].to_numpy()
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
for col, lab, c, m in [
    ("ARI_ct", "ARI vs cell-type niches", "#0072B2", "o"),
    ("NMI_ct", "NMI vs cell-type niches", "#0072B2", "s"),
    ("ARI_mol", "ARI vs molecular niches", "#D55E00", "o"),
    ("NMI_mol", "NMI vs molecular niches", "#D55E00", "s"),
]:
    axes[0].plot(
        x,
        res[col],
        marker=m,
        color=c,
        ls="-" if col.startswith("ARI") else "--",
        label=lab,
    )
axes[0].set_xlabel("layer_ratio (0 = plain Leiden)")
axes[0].set_ylabel("agreement")
axes[0].set_ylim(0, None)
axes[0].legend(frameon=False, fontsize=7)
axes[1].plot(x, res["coherence"], marker="o", color="#009E73", label="observed")
axes[1].plot(
    x,
    res["coherence_shuffled"],
    marker="o",
    color="#999999",
    ls="--",
    label="labels shuffled",
)
axes[1].axhline(
    results.loc["authors: cell-type niches", "coherence"],
    color="#CC79A7",
    lw=1,
    ls=":",
    label="authors' cell-type niches",
)
axes[1].set_xlabel("layer_ratio (0 = plain Leiden)")
axes[1].set_ylabel("same-label neighbours")
axes[1].set_ylim(0, 1)
axes[1].legend(frameon=False, fontsize=7)
for ax in axes:
    ax.set_xticks(x)
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(OUT / "fig4_metrics_vs_layer_ratio.png", dpi=150)

### What we see

- **Spatial coherence rises steadily with `layer_ratio`**: from 0.58 for plain Leiden to 0.66, 0.88 and 0.94 at ratios 0.5, 1 and 2, while the shuffled baseline stays at 0.14–0.22. Even plain Leiden is more coherent than the authors' cell-type niches (0.45), so the tissue itself carries much of the spatial structure.
- **Agreement with the authors' niches changes little and then drops.** Against the cell-type niches, ARI moves from 0.21 (Leiden) to 0.23 (ratio 0.5) and then down to 0.18–0.19; against the molecular niches it falls from 0.21 to 0.07. This is expected: the authors' niches were built without coordinates and are scattered, so large smooth domains cannot match them spot by spot. Low agreement here does not mean the domains are wrong.
- **What a good `layer_ratio` is cannot be read from this table alone.** Between 0.5 and 1 the maps change from "expression clusters with less noise" to "large tissue regions"; which one is useful depends on the question. The preprint used 0.8 for single Visium slides.
- **Caveat:** this is one slide, one seed and one target number of clusters. Treat the numbers as an illustration of the trade-off, not as a benchmark.

Finally, the total run time of the notebook.

In [ ]:
print(f"total runtime: {time.time() - t_start:.0f} s")

## 7. A systematic check: how much space, and how many domains?

Sections 5–6 compared plain Leiden with SpatialLeiden at three `layer_ratio` values on one slide and one seed. The rest of this notebook turns that into a controlled experiment (**experiment 0**): eight `layer_ratio` values, five seeds, three slides, and four choices of the number of domains K. The design and its pre-specified interpretation are in `exp0_layer_ratio_sweep/README.md`.

**Two terms used from here on:**

- **`layer_ratio`:** SpatialLeiden maximises *quality = Q(expression graph) + `layer_ratio` × Q(spatial graph)* for one shared partition. It sets how much a spot is rewarded for sharing a cluster with its physical neighbours, relative to its expression neighbours.
- **K:** the number of domains each map must have. It is **not** the k of the nearest-neighbour graph, which stays at 15. K is held fixed while `layer_ratio` changes, because fewer, larger clusters are always more contiguous, which would confuse the measurement.

**Slides:**

- P9 IZ rep2 (`ACH0013`), the slide used so far.
- P9 IZ (`ACH0012`): same patient and lesion, typical depth.
- P1 control (`10X001`): the negative control, with little visible structure.

The cell below defines the loading and preprocessing used for every slide: the same steps as sections 1 and 3, packaged as functions. It then rebuilds this slide with the neighbour-graph seed fixed at 0, as in `exp0_sweep.py`, so the numbers here match the full sweep exactly.

In [ ]:
import json
import subprocess
import sys
import time
import warnings
from itertools import combinations
from pathlib import Path

import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import spatialleiden as sl
import squidpy as sq
from matplotlib.collections import EllipseCollection
from scipy.spatial import cKDTree
from sklearn.metrics import adjusted_rand_score

warnings.filterwarnings("ignore", category=FutureWarning)
DM = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "Derived_Modalities"
)
DATA = DM / "data" / "kuppe_2022"
RES = DM / "results" / "kuppe_2022"
SCRIPT = (
    DM / "03_single_modality_clustering" / "exp0_layer_ratio_sweep" / "exp0_sweep.py"
)

SLIDES = {
    "ACH0013": ("Visium_GT_IZ_P9_rep2.h5ad", "P9 IZ rep2 (ACH0013)"),
    "ACH0012": ("Visium_GT_IZ_P9.h5ad", "P9 IZ (ACH0012)"),
    "10X001": ("Visium_control_P1.h5ad", "P1 control (10X001)"),
}
DEMO = "ACH0013"  # slide used for the step-by-step part
RATIOS = [0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 2.5]
SEEDS = [0, 1, 2, 3, 4]
DIRECTED = (
    False,
    False,
)  # symmetric expression kNN graph, regular Visium grid (preprint code)
N_SEARCH = 30  # resolution-search iterations (package default 15)
GRAPH_SEED = 0  # PCA and neighbour graph: fixed for the whole sweep
SLIDE_COL = {"ACH0013": "#D55E00", "ACH0012": "#E69F00", "10X001": "#0072B2"}
print(
    "spatialleiden",
    sl.__version__,
    "| scanpy",
    sc.__version__,
    "| squidpy",
    sq.__version__,
)
print("project folder:", DM)


def load_slide(run):
    h5 = ad.read_h5ad(DATA / "processed1" / SLIDES[run][0])
    spots = pd.read_csv(DATA / "visium_tables" / f"{run}_spots.csv").set_index(
        "barcode"
    )
    assert set(spots.index) == set(h5.obs_names), f"{run}: barcodes differ"
    spots = spots.loc[h5.obs_names]  # explicit join by barcode
    a = ad.AnnData(
        X=h5.raw.X.copy(),
        obs=pd.DataFrame(index=h5.obs_names),
        var=pd.DataFrame(index=h5.raw.var_names),
    )
    assert np.allclose(
        np.asarray(a.X.sum(1)).ravel(), spots.nCount_Spatial.to_numpy()
    ), "counts mismatch"
    a.layers["counts"] = a.X.copy()
    a.obsm["spatial"] = spots[["pxl_col_fullres", "pxl_row_fullres"]].to_numpy(float)
    for c in ["array_row", "array_col", "celltype_niche", "molecular_niche"]:
        a.obs[c] = spots[c].to_numpy()
    return a, spots


def preprocess(a):
    sc.pp.filter_genes(a, min_cells=10)
    sc.pp.highly_variable_genes(a, n_top_genes=2000, flavor="seurat_v3", layer="counts")
    sc.pp.normalize_total(a)
    sc.pp.log1p(a)
    sc.pp.scale(a, zero_center=False)
    sc.pp.pca(a, n_comps=30, mask_var="highly_variable", random_state=GRAPH_SEED)
    sc.pp.neighbors(
        a, n_neighbors=15, n_pcs=30, random_state=GRAPH_SEED
    )  # expression graph
    sq.gr.spatial_neighbors(a, coord_type="grid", n_neighs=6)  # spatial graph



def search_resolution_steps(n_clusters_at, target, start=1.0, step=0.1, n_iter=N_SEARCH):
    """Step-halving search for a resolution that gives `target` clusters.

    Same algorithm as spatialleiden 0.4's private `_resolution_search._search_resolution`,
    copied here so this notebook does not depend on a private API (it is not guaranteed
    to stay the same between versions). It is needed because no public function searches
    the expression resolution with `spatialleiden(layer_ratio=0)` itself:
    `search_resolution_latent` uses plain Leiden, a different optimiser path (section 10).
    Returns (resolution, reached_target).
    """
    res = start
    n = n_clusters_at(res)
    i = 1
    while n != target:
        if i >= n_iter:
            return res, False
        sign = 1 if n < target else -1
        res += step * sign
        n = n_clusters_at(res)
        if (1 if n < target else -1) != sign:
            step /= 2
        i += 1
    return res, True


adata, spots = load_slide(DEMO)
preprocess(adata)
print(
    f"{SLIDES[DEMO][1]}: {adata.n_obs} spots, {adata.n_vars} genes, graphs rebuilt with seed {GRAPH_SEED}"
)

## 8. Measures

Every partition is scored the same way:

- **Spatial coherence:** the share of neighbouring spot pairs (edges of the spatial graph) whose two spots carry the same label. This is the edge-weighted join-count form, the same definition as in section 6. Interior spots with six neighbours contribute six pairs, edge spots fewer.
- **Coherence gap:** coherence minus the same quantity after shuffling the labels within the slide. The shuffle is the mean of 20 permutations, from a fresh generator with a fixed seed per call, so the null does not depend on run order. Raw coherence depends on cluster sizes and slide shape; only the gap above shuffling can be interpreted.
- **ARI to ratio 0:** agreement with the no-space partition of the same seed, i.e. how far space moved the result.
- **Seed stability:** mean pairwise ARI among the seeds at one setting. The PCA and both graphs are built once with a fixed seed, so this captures **only Leiden's randomness**, not variability in graph construction. Read it as a lower bound on run-to-run variability, not as overall reproducibility.

In [ ]:
EI, EJ = adata.obsp["spatial_connectivities"].nonzero()
SHUFFLE_SEED = 0


def coherence(labels, i=EI, j=EJ):
    """Edge-weighted: share of neighbouring spot pairs with the same label."""
    labels = np.asarray(labels)
    return float((labels[i] == labels[j]).mean())


def coherence_gap(labels, n_shuffle=20, i=EI, j=EJ):
    """Observed coherence, mean coherence after shuffling (fresh generator, fixed seed), gap."""
    rng = np.random.default_rng(SHUFFLE_SEED)
    obs = coherence(labels, i, j)
    shuf = float(
        np.mean(
            [coherence(rng.permutation(np.asarray(labels)), i, j) for _ in range(n_shuffle)]
        )
    )
    return obs, shuf, obs - shuf


ct = adata.obs.celltype_niche.to_numpy()
print(
    "authors' cell-type niches on this slide: coherence %.2f, shuffled %.2f, gap %.2f"
    % coherence_gap(ct)
)

## 9. Three ways to choose K

| Choice | Rule | Why |
|---|---|---|
| **K = 15** | Plain Leiden at the package default (resolution 1.0, seed 0) on the reference slide `ACH0013`, held on all slides | The experiment brief's rule: take K from the ratio-0 run |
| **K = 9** | Fixed | Number of the authors' cell-type niches overall; used in the intro notebook |
| **K per slide** | Number of the authors' cell-type (or molecular) niches present on that slide | To check whether the low agreement with the published niches is only a matter of cluster number |

The cell below computes each choice for every slide. Plain Leiden at resolution 1.0 is run on every slide, to show that the "natural" number of clusters differs between slides.

In [ ]:
k_table = []
for run in SLIDES:
    a, _ = (adata, spots) if run == DEMO else load_slide(run)
    if run != DEMO:
        preprocess(a)
    sl.leiden(a, resolution=1.0, directed=False, random_state=0, key_added="probe")
    k_table.append(
        {
            "slide": SLIDES[run][1],
            "Leiden clusters at resolution 1.0": a.obs["probe"].nunique(),
            "authors' cell-type niches": a.obs.celltype_niche.nunique(),
            "authors' molecular niches": a.obs.molecular_niche.nunique(),
        }
    )
k_table = pd.DataFrame(k_table).set_index("slide")
k_table

**How to read this:**

- **The brief's K = 15** comes from the first row: plain Leiden on `ACH0013` at resolution 1.0.
- **The other slides** give fewer clusters at the same resolution, and the authors have fewer niches on every slide.
- **The per-slide choices** take K from the last two columns.

## 10. Sanity check: is `layer_ratio = 0` the same as plain Leiden?

At `layer_ratio = 0` the spatial layer adds nothing to the objective, so we would expect plain Leiden's result. We compare three runs at the same resolution and seed:

1. Plain `sl.leiden` on the expression graph.
2. `sl.multiplex_leiden` with **only** the expression layer: the same objective and the same optimiser as SpatialLeiden.
3. `sl.spatialleiden(layer_ratio=0)`: both layers, the spatial one at weight 0.

`multiplex_leiden` is given `layer_weights=1.0` because its default (`1`, an integer) raises a `TypeError` in this version.

In [ ]:
rows = []
for seed in [0, 1, 2]:
    sl.leiden(
        adata, resolution=1.0, directed=False, random_state=seed, key_added="plain"
    )
    latent_only = sl.multiplex_leiden(
        adata.obsp["connectivities"],
        directed=False,
        layer_weights=1.0,
        partition_kwargs={"resolution_parameter": 1.0},
        random_state=seed,
    )
    sl.spatialleiden(
        adata,
        resolution=(1.0, 1.0),
        layer_ratio=0,
        directed=DIRECTED,
        n_iterations=-1,
        random_state=seed,
        key_added="ratio0",
    )
    rows.append(
        {
            "seed": seed,
            "K plain Leiden": adata.obs.plain.nunique(),
            "K ratio 0": adata.obs.ratio0.nunique(),
            "ARI plain vs latent-only multiplex": adjusted_rand_score(
                adata.obs.plain, latent_only
            ),
            "ARI plain vs spatialleiden ratio 0": adjusted_rand_score(
                adata.obs.plain, adata.obs.ratio0
            ),
        }
    )
plain_by_seed = {}
for seed in SEEDS:
    sl.leiden(adata, resolution=1.0, directed=False, random_state=seed, key_added="p")
    plain_by_seed[seed] = adata.obs.p.to_numpy()
noise = np.mean(
    [
        adjusted_rand_score(plain_by_seed[a], plain_by_seed[b])
        for a, b in combinations(SEEDS, 2)
    ]
)
print(f"plain Leiden, seed against seed (5 seeds): mean ARI {noise:.2f}")
pd.DataFrame(rows).round(3)

**How to read this:**

- **The objective is the same.** Plain Leiden and latent-only multiplex agree exactly (ARI = 1).
- **The run is not.** Adding the spatial layer at weight 0 changes the result (ARI about 0.7–0.8), sometimes even the cluster count. The zero-weight layer contributes nothing to the score, but leidenalg's multiplex optimiser still considers moves along its edges. It therefore searches differently and ends in a different local optimum.
- **The difference is no larger than seed noise.** Plain Leiden agrees with *itself* across seeds at about the printed mean ARI, so ratio 0 is a valid no-space baseline.
- **Consequence for the design:** the baseline must be `spatialleiden(layer_ratio=0)` itself, with its own resolution search. Every ratio, including 0, then uses the same optimiser.

## 11. Holding K fixed across the sweep, shown on one slide and one seed

- **At `layer_ratio = 0`:** search the *expression* resolution that gives K, using `spatialleiden(layer_ratio=0)` itself. The package's `search_resolution_latent` would search with plain Leiden instead (section 10).
- **At `layer_ratio > 0`:** keep that expression resolution and search the *spatial* resolution with `sl.search_resolution_spatial`.

Both searches use the same step-halving routine as the package, with up to 30 iterations.

In [ ]:
def sweep_one_seed(a, k, seed, ei, ej):
    # ratio 0: expression resolution giving k, searched with spatialleiden(layer_ratio=0)
    def n_ratio0(res):
        sl.spatialleiden(
            a,
            resolution=(res, 1.0),
            layer_ratio=0,
            directed=DIRECTED,
            n_iterations=-1,
            random_state=seed,
            key_added="tmp",
        )
        return a.obs["tmp"].nunique()

    r_lat, _ = search_resolution_steps(n_ratio0, k)
    rows, labels = [], {}
    for ratio in RATIOS:
        key = f"r{ratio}_s{seed}"
        if ratio == 0:
            r_sp = np.nan
            sl.spatialleiden(
                a,
                resolution=(r_lat, 1.0),
                layer_ratio=0,
                directed=DIRECTED,
                n_iterations=-1,
                random_state=seed,
                key_added=key,
            )
        else:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                r_sp = sl.search_resolution_spatial(
                    a,
                    k,
                    n_iter=N_SEARCH,
                    resolution=(r_lat, 1.0),
                    layer_ratio=ratio,
                    directed=DIRECTED,
                    n_iterations=-1,
                    random_state=seed,
                    key_added=key,
                )
            sl.spatialleiden(
                a,
                resolution=(r_lat, r_sp),
                layer_ratio=ratio,
                directed=DIRECTED,
                n_iterations=-1,
                random_state=seed,
                key_added=key,
            )  # the run at the found resolution
        lab = a.obs[key].astype(str).to_numpy()
        labels[ratio] = lab
        obs, shuf, gap = coherence_gap(lab, i=ei, j=ej)
        rows.append(
            {
                "layer_ratio": ratio,
                "K": len(set(lab)),
                "res_expression": r_lat,
                "res_spatial": r_sp,
                "coherence": obs,
                "shuffled": shuf,
                "gap": gap,
                "ARI_to_ratio0": adjusted_rand_score(labels[0], lab),
                "ARI_authors_celltype_niches": adjusted_rand_score(
                    a.obs.celltype_niche, lab
                ),
            }
        )
    return pd.DataFrame(rows), labels


t0 = time.time()
demo_table, demo_labels = sweep_one_seed(adata, k=9, seed=0, ei=EI, ej=EJ)
print(f"{DEMO}, K = 9, seed 0: {time.time() - t0:.0f} s")
demo_table.round(3)

**How to read this:**

- **The `K` column** should show 9 in every row. That confirms the cluster count was held fixed.
- **The spatial resolution needed** changes with `layer_ratio`. This trajectory shows how hard the search works to keep K.
- **The gap rises and ARI to ratio 0 falls** as space gets more weight.

In [ ]:
def draw_spots(ax, xy, colours, title):
    x, y = xy[:, 0], xy[:, 1]
    pitch = np.median(cKDTree(xy).query(xy, k=2)[0][:, 1])
    d = np.full(len(x), 1.05 * pitch)
    coll = EllipseCollection(
        d,
        d,
        np.zeros(len(x)),
        units="xy",
        offsets=xy,
        offset_transform=ax.transData,
        facecolors=colours,
        linewidths=0,
    )
    coll.set_rasterized(True)
    ax.add_collection(coll)
    ax.set_xlim(x.min() - pitch, x.max() + pitch)
    ax.set_ylim(y.max() + pitch, y.min() - pitch)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(title, fontsize=9)


def match_to(ref, lab):
    # recolour clusters to the ratio-0 cluster they overlap most (Hungarian matching), for display only
    tab = pd.crosstab(pd.Series(lab), pd.Series(ref))
    r, c = linear_sum_assignment(-tab.to_numpy())
    mapping = {tab.index[i]: tab.columns[j] for i, j in zip(r, c)}
    nxt = len(tab.columns)
    for k_ in tab.index:
        if k_ not in mapping:
            mapping[k_] = f"extra{nxt}"
            nxt += 1
    return pd.Series(lab).map(mapping).to_numpy()


PAL = [c for i, c in enumerate(mpl.colormaps["tab20"].colors) if i not in (14, 15)]
show = [0, 0.5, 1.0, 2.0]
ref = demo_labels[0]
cats = sorted(set(ref), key=lambda s: int(s) if s.isdigit() else 99)
col = {c: PAL[i % len(PAL)] for i, c in enumerate(cats)}
fig, axes = plt.subplots(1, len(show), figsize=(16, 4.2))
for ax, ratio in zip(axes, show):
    lab = ref if ratio == 0 else match_to(ref, demo_labels[ratio])
    draw_spots(
        ax,
        adata.obsm["spatial"],
        [col.get(v, "#777777") for v in lab],
        f"layer_ratio {ratio}",
    )
fig.suptitle(f"{SLIDES[DEMO][1]}: K = 9 domains, seed 0 (colours matched to ratio 0)")
plt.show()

**How to read this:**

- **At 0:** clusters are scattered spot by spot.
- **At 0.5:** they consolidate into regions that follow the tissue. There is a narrow domain along the vSMC-lined vessel, the myocardium at the top, and the fibroblast- and myeloid-rich lesion at the lower right. Compare with the cell2location maps in the slide atlas.
- **From about 1–2:** the slide is cut into compact, tile-like patches, whatever the tissue looks like.

## 12. The full sweep: 3 slides × 8 ratios × 5 seeds, for four K settings

The script `exp0_sweep.py` contains the same functions as sections 7–11. It runs one process per slide × seed, because Leiden uses one CPU core. That takes about 1–2 minutes per K setting instead of an hour.

- **If results already exist** in `results/kuppe_2022/`, they are loaded.
- **To recompute,** set `RUN_FULL_SWEEP = True`.

In [ ]:
RUN_FULL_SWEEP = False
SETTINGS = {  # label -> (output folder, script arguments)
    "K = 15 (brief)": ("exp0", ["--k", "15"]),
    "K = 9": ("exp0_k9", ["--k", "9"]),
    "K = authors' cell-type niches": (
        "exp0_k_celltype_niche",
        ["--k-from", "celltype_niche"],
    ),
    "K = authors' molecular niches": (
        "exp0_k_molecular_niche",
        ["--k-from", "molecular_niche"],
    ),
}
for label, (folder, args) in SETTINGS.items():
    out = RES / folder
    if RUN_FULL_SWEEP or not (out / "exp0_sweep.csv").exists():
        out.mkdir(parents=True, exist_ok=True)
        t0 = time.time()
        subprocess.run(
            [
                sys.executable,
                "-W",
                "ignore",
                str(SCRIPT),
                str(DATA),
                str(out),
                *args,
                "--jobs",
                "15",
            ],
            check=True,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        print(f"{label}: computed in {time.time() - t0:.0f} s")
    else:
        print(f"{label}: loaded from {out.relative_to(DM)}")

sweep = pd.concat(
    [
        pd.read_csv(RES / f / "exp0_sweep.csv").assign(setting=lab)
        for lab, (f, _) in SETTINGS.items()
    ]
)
sanity = pd.concat(
    [
        pd.read_csv(RES / f / "exp0_sanity_ratio0.csv").assign(setting=lab)
        for lab, (f, _) in SETTINGS.items()
    ]
)
print(
    len(sweep),
    "runs;",
    int((sweep.n_clusters != sweep.target_k).sum()),
    "did not reach K exactly (flagged in search_failed)",
)
# runs whose search missed K stay in the CSV files but are left out of every mean below:
# fewer, larger clusters are mechanically more coherent (the confound fixed K removes)
ok = sweep[sweep.n_clusters == sweep.target_k]
missed = (
    sweep.assign(missed=sweep.n_clusters != sweep.target_k)
    .groupby(["setting", "slide", "layer_ratio"])
    .missed.sum()
)
print("runs that missed K (setting, slide, layer_ratio: count):")
print(missed[missed > 0].to_string() if (missed > 0).any() else "none")
sweep.groupby(["setting", "slide"]).target_k.first().unstack()

**How to read this:** the table shows the K used for every slide under each setting.

Runs whose search did not reach K within 30 iterations are listed above. They stay in the CSV files but are **left out of every mean and of the seed stability below**. A run with fewer, larger clusters is more coherent simply because of its size, which is exactly the confound that holding K fixed is meant to remove.

In [ ]:
fig, axes = plt.subplots(3, len(SETTINGS), figsize=(18, 10), sharex=True)
metrics = [
    ("coherence_gap", "Coherence above shuffled"),
    ("ARI_to_ratio0", "ARI to ratio-0 partition"),
    ("seed_ARI_mean", "Seed stability (mean pairwise ARI)"),
]
for j, (label, _) in enumerate(SETTINGS.items()):
    d = ok[ok.setting == label]  # only runs that reached K
    for i, (m, name) in enumerate(metrics):
        ax = axes[i, j]
        for run in SLIDES:
            g = d[d.slide == run].groupby("layer_ratio")[m]
            ax.fill_between(
                g.mean().index, g.min(), g.max(), color=SLIDE_COL[run], alpha=0.2, lw=0
            )
            ax.plot(
                g.mean().index,
                g.mean(),
                "-o",
                ms=3,
                color=SLIDE_COL[run],
                label=SLIDES[run][1],
            )
        if i == 0:
            ax.set_title(label, fontsize=10)
        if j == 0:
            ax.set_ylabel(name)
        if i == 2:
            ax.set_xlabel("layer_ratio")
        if m != "coherence_gap":
            ax.set_ylim(0, 1.02)
axes[0, 0].legend(frameon=False, fontsize=8)
fig.suptitle("Experiment 0: line = mean over 5 seeds, band = range over seeds", y=1.0)
fig.tight_layout()
plt.show()

**How to read this:** columns are the four K settings; rows are the three measures; colours are the slides.

- **The shapes are the same in every column.** The conclusions do not depend on K.
- **Coherence (top row):** it rises between `layer_ratio` 0.25 and 1.0, and saturates above about 1.5 on **all** slides, the control included.
- **Distance from the expression baseline (middle row):** ARI to ratio 0 falls fastest on P1, the control.
- **Stability (bottom row):** P1 is the least stable without space. At high ratios it becomes stable, because space imposes the same tiling regardless of the seed.

In [ ]:
summary = (
    ok[ok.layer_ratio.isin([0, 0.5, 1.0])]
    .groupby(["slide", "setting", "layer_ratio"])
    .agg(
        K=("target_k", "first"),
        gap=("coherence_gap", "mean"),
        ARI_to_ratio0=("ARI_to_ratio0", "mean"),
        seed_stability=("seed_ARI_mean", "first"),
        ARI_authors_celltype=("ARI_celltype_niche", "mean"),
        ARI_authors_molecular=("ARI_molecular_niche", "mean"),
    )
    .round(2)
)
summary

**How to read this:**

- **Agreement with the authors' niches stays low under every K,** at most about 0.2 on P9 and about 0.04 on P1, even when K equals their niche count. The difference is therefore not about cluster number. Their niches group spots by cell-type composition (or pooled expression) across 27 slides, without coordinates; ours use one slide's expression plus space. They are a reference, not ground truth.
- **Disease and control differ before space is added:** the gap at ratio 0 is clearly higher on both P9 sections than on P1.

In [ ]:
print("Sanity check per slide and setting (mean over seeds):")
(
    sanity.groupby(["setting", "slide"])[
        [
            "ARI_leiden_vs_latent_only_multiplex",
            "ARI_leiden_vs_spatialleiden_ratio0",
            "plain_leiden_seed_ARI_mean",
        ]
    ]
    .mean()
    .round(2)
)

**How to read this:** for every slide and setting, the ratio-0-vs-Leiden ARI (middle column) is no lower than plain Leiden's seed-to-seed ARI (right column). The ratio-0 baseline differs from plain Leiden by no more than seed noise.

## 13. Domain maps for all slides (K = authors' cell-type niches, seed 0)

In [ ]:
folder = "exp0_k_celltype_niche"
fig, axes = plt.subplots(len(SLIDES), 5, figsize=(18, 3.8 * len(SLIDES)))
NICHE_COL = {
    f"ctniche_{i}": c
    for i, c in enumerate(
        [
            "#D51F26",
            "#272E6A",
            "#208A42",
            "#89288F",
            "#F47D2B",
            "#FEE500",
            "#8A9FD1",
            "#C06CAB",
            "#D8A767",
        ],
        start=1,
    )
}
for i, run in enumerate(SLIDES):
    lab = pd.read_csv(RES / folder / f"labels_{run}.csv.gz", index_col=0, dtype=str)
    xy = lab[["x", "y"]].astype(float).to_numpy()
    ref = lab["r0_s0"].to_numpy()
    cats = sorted(set(ref), key=int)
    col = {c: PAL[k % len(PAL)] for k, c in enumerate(cats)}
    for j, ratio in enumerate([0, 0.5, 1.0, 2.0]):
        key = "r0_s0" if ratio == 0 else f"r{ratio}_s0"
        m = ref if ratio == 0 else match_to(ref, lab[key].to_numpy())
        draw_spots(
            axes[i, j], xy, [col.get(v, "#777777") for v in m], f"layer_ratio {ratio}"
        )
    draw_spots(
        axes[i, 4],
        xy,
        [NICHE_COL[v] for v in lab.celltype_niche],
        "authors' cell-type niches",
    )
    axes[i, 0].text(
        -0.05,
        0.5,
        SLIDES[run][1],
        transform=axes[i, 0].transAxes,
        rotation=90,
        ha="right",
        va="center",
        fontsize=10,
    )
plt.tight_layout()
plt.show()

**How to read this:**

- **P9 (both sections):** at about 0.5, the domains follow tissue structures: the vessel, the myocardium and the lesion. Above about 1 they turn into compact tiles.
- **P1 (control):** without space there is no clear structure. From about 0.75 the slide is tiled into patches that match nothing on the H&E or the cell2location maps. This is over-smoothing.
- **Comparison with the paper:** the authors' P1 niches are themselves close to spatial noise.

## 14. Conclusions

1. **The baseline is valid.** `spatialleiden(layer_ratio=0)` differs from plain Leiden by no more than seed noise, as long as it is produced by the same function and resolution search as the other ratios.
2. **Coherence rises meaningfully between `layer_ratio` 0.25 and 1.0,** and saturates above about 1.5 on every slide, the control included. Coherence alone therefore cannot separate real structure from structure imposed by the spatial term.
3. **Space helps on the disease slide around `layer_ratio` ≈ 0.5.** Domains are contiguous, still related to expression, and follow visible tissue. Above about 1 it over-smooths.
4. **On the control, space over-smooths from about 0.75.**
5. **The choice of K changes details, not conclusions.** Matching K to the authors' niche counts barely changes agreement with their niches.

**Working range for the next experiments:** `layer_ratio` ≈ 0.5–0.75, single slides, the RNA layer plus one derived modality (TF activity on P9; see `02_modalities/collectri/`).

## 15. What comes next in project C

- **Derived modalities as extra layers.** Instead of one expression layer plus space, `spatialleiden_multimodal` takes several feature layers (for example pathway activity from PROGENy, transcription-factor activity from CollecTRI, cell-cell signalling from LIANA+, or the shipped cell2location proportions) plus the spatial layer, each with its own weight. Step 03 clusters each modality alone with the same recipe as here; step 04 combines them over a grid of layer weights.
- **Multi-slide.** Here we used one slide. Later we will cluster several sections together (one spatial graph per section, batch-corrected expression), as in the package's multi-sample guide (`SpatialLeiden/docs/source/guides/multisample.ipynb`).
- **Better references.** Section-level zones and patient groups are the independent readouts for evaluation; the authors' niches are useful but were themselves built from the expression and composition data.